# Week 38, Day 1: The retina: centre and surround, ON and OFF, adaptation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The area-response curve

Write dog(n, sc, ss, w), returning an n by n difference-of-Gaussians kernel on a grid of 0.1° pixels centred on the middle pixel (each Gaussian normalised to sum to 1 on the grid). Then compute the response to centred spots of increasing diameter as the sum of the kernel over the spot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 101                                         # 10 degrees at 0.1 degree per pixel
xy = (np.arange(n) - n // 2) * 0.1
X, Y = np.meshgrid(xy, xy); R = np.hypot(X, Y)

def dog(n, sc, ss, w):
    centre = ...
    surround = ...
    return centre / centre.sum() - w * surround / surround.sum()

K = dog(n, sc=0.25, ss=1.0, w=0.85)
diams = np.arange(0.1, 8.01, 0.1)
resp = np.array([K[R <= d / 2].sum() for d in diams])
print(f"best spot diameter {diams[np.argmax(resp)]:.1f} deg; response to the largest spot is {resp[-1] / resp.max():.0%} of the peak")
plt.plot(diams, resp); plt.xlabel("spot diameter (deg)"); plt.ylabel("response"); plt.show()

In [ ]:
import numpy as np
def test_kernel():
    "the kernel's centre is positive and its sum is small but positive"
    assert K[n // 2, n // 2] > 0 and 0 < K.sum() < 0.3

def test_curve():
    "the response peaks at an intermediate spot size and falls below 40% of the peak for the largest spot"
    i = int(np.argmax(resp))
    assert 0 < i < len(diams) - 1 and resp[-1] < 0.4 * resp.max()

test_kernel()
test_curve()
print("All checks passed")

### Exercise: Filtering a step edge

Make a 1D DoG profile (the same σ values, on a line of 0.1° pixels), convolve it with a step of light from 0 to 1 at the middle of a 400-pixel line ('same' mode), and split the result into ON = max(out, 0) and OFF = max(-out, 0).

In [ ]:
x1 = (np.arange(121) - 60) * 0.1
k1 = np.exp(-x1 ** 2 / (2 * 0.25 ** 2)); k1 /= k1.sum()
s1 = np.exp(-x1 ** 2 / (2 * 1.0 ** 2)); s1 /= s1.sum()
kernel = k1 - 0.85 * s1
line = np.r_[np.zeros(200), np.ones(200)]
out = ...
on = ...
off = ...
print(f"peak ON response {on.max():.3f} at pixel {np.argmax(on)}, peak OFF {off.max():.3f} at pixel {np.argmax(off)} (edge at 200)")
plt.plot(line * 0.2, "k", lw=0.8, label="light (scaled)"); plt.plot(on, label="ON"); plt.plot(-off, label="OFF (negative)"); plt.xlim(120, 280); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_flat():
    "far from the edge, the response is 0 on the dark side and only the small leak 1 - w = 0.15 on the bright side"
    assert abs(out[100]) < 0.01 and abs(out[300] - 0.15) < 0.01

def test_overshoot():
    "the ON response peaks just on the bright side of the edge and the OFF response just on the dark side"
    edge = len(line) // 2
    assert edge <= np.argmax(on) < edge + 15 and edge - 15 < np.argmax(off) < edge

test_flat()
test_overshoot()
print("All checks passed")

### Exercise: Contrast invariance by divisive gain

For backgrounds from 0.01 to 10,000 (log spaced), compute the response to a flash of 10% contrast (I = 1.1 I_b) with the adapting model, and with a non-adapting linear model r = I - I_b.

In [ ]:
I0 = 1.0
backgrounds = np.logspace(-2, 4, 25)
resp_adapt = ...
resp_linear = ...
plt.loglog(backgrounds, resp_adapt, "o-", label="adapting"); plt.loglog(backgrounds, resp_linear, "s-", label="linear")
plt.xlabel("background intensity"); plt.ylabel("response to a 10% flash"); plt.legend(); plt.show()
print("adapting responses at the brightest five backgrounds:", np.round(resp_adapt[-5:], 4))

In [ ]:
import numpy as np
def test_weber():
    "for backgrounds at least 100 times I0, responses to 10% contrast vary by less than 2%"
    big = resp_adapt[backgrounds >= 100 * I0]
    assert big.max() / big.min() < 1.02

def test_linear():
    "without adaptation, the response to 10% contrast grows over the range by a factor above 1000"
    assert resp_linear[-1] / resp_linear[0] > 1000

test_weber()
test_linear()
print("All checks passed")